In [2]:
from pathlib import Path
import pandas as pd
import tarfile
import urllib.request

def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing = load_housing_data()
housing.head(1)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY


In [8]:
# 훈련세트 테스트세트 분할
from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(housing, test_size=0.2,random_state=42)

# 이상치 탐지 라이브러리
# 정상데이터보다 이상치는 더 쉽게(적은 분할로) 고립된다
# 랜던하게 feature선택 랜덤하게 split value 선택 -> 계속.. 어떤 데이터가 얼마나 빨리 고립되는가
# 트리알고리즘, 이상치는 적은 분할회수로고립, 트리가 깊지 않음
from sklearn.ensemble import IsolationForest
isolation_forest = IsolationForest(random_state=42)
housing_numeric_for_outlier = train_set.drop('ocean_proximity',axis=1).dropna()
housing_numeric_for_outlier.head(2)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
14196,-117.22,32.75,34.0,6001.0,1111.0,2654.0,1072.0,4.5878,291000.0
8267,-117.03,32.69,10.0,901.0,163.0,698.0,167.0,4.6648,156100.0


In [ ]:
outlier_pred = pd.Series(1,index = train_set.index)
outlier_pred.loc[housing_numeric_for_outlier.index] = isolation_forest.fit_predict(housing_numeric_for_outlier)
# 정상 1 이상치 -1
